## 1. Load and Preprocess WBC (original)

In [1]:
import sys
sys.path.append("../")
from utils.fetch_bcw_dataset import fetch_local_bcw
import pandas as pd

In [2]:
bcw_df = fetch_local_bcw("../data/Breast Cancer Wisconsin.csv")

In [3]:
print(bcw_df.isna().any())
print(f"\nTotal na line:\n{bcw_df.isna().sum()}")

Clump_thickness                False
Uniformity_of_cell_size        False
Uniformity_of_cell_shape       False
Marginal_adhesion              False
Single_epithelial_cell_size    False
Bare_nuclei                     True
Bland_chromatin                False
Normal_nucleoli                False
Mitoses                        False
Class                          False
dtype: bool

Total na line:
Clump_thickness                 0
Uniformity_of_cell_size         0
Uniformity_of_cell_shape        0
Marginal_adhesion               0
Single_epithelial_cell_size     0
Bare_nuclei                    16
Bland_chromatin                 0
Normal_nucleoli                 0
Mitoses                         0
Class                           0
dtype: int64


In [4]:
nona_bcw_df = bcw_df[~bcw_df['Bare_nuclei'].isna()]

In [5]:
nona_bcw_df.dtypes

Clump_thickness                  int64
Uniformity_of_cell_size          int64
Uniformity_of_cell_shape         int64
Marginal_adhesion                int64
Single_epithelial_cell_size      int64
Bare_nuclei                    float64
Bland_chromatin                  int64
Normal_nucleoli                  int64
Mitoses                          int64
Class                            int64
dtype: object

In [6]:
from src.preprocessing import foolish_discretization

In [7]:
discretized_bare_nuclei = foolish_discretization(nona_bcw_df['Bare_nuclei'])
assert discretized_bare_nuclei.tolist() == discretized_bare_nuclei.copy().astype("int64").tolist()

In [8]:
clear_bcw_df = fetch_local_bcw("../data/Clear Breast Cancer Wisconsin.csv")
clear_bcw_df.dtypes

Clump_thickness                int64
Uniformity_of_cell_size        int64
Uniformity_of_cell_shape       int64
Marginal_adhesion              int64
Single_epithelial_cell_size    int64
Bare_nuclei                    int64
Bland_chromatin                int64
Normal_nucleoli                int64
Mitoses                        int64
Class                          int64
dtype: object

## 2. Reduction and Calculation of Core

In [9]:
import numpy as np

In [77]:
U = clear_bcw_df
C = list(U.columns[0:-1])
V = [c for c in U.columns if c not in C]
D = U[V].drop_duplicates().to_numpy()

The following codes aim to debug.
While you replace the statement in body of under_appr: 
`return X.loc[list(under_appr_ind_set)]` into 
`return X.iloc[list(under_appr_ind_set)]`,
you will encounter IndexError.

```python
under_appr_ind_set = set()
for _, D_i in U_div_D.items():
    D_1 = D_i
    break
X_ind_set = set(D_1.index)
for cls_set in equiv_cls.values():
    if not cls_set.issubset(X_ind_set): continue
    under_appr_ind_set = under_appr_ind_set.union(cls_set)
for i in list(under_appr_ind_set):
    try:
        D_1.loc[[i]]
        # print(f"{i} works!")
    except IndexError as ie:
        print(f"IndexError: {i}")
```

In [78]:
def equiv_cls(B):
    equiv_cls_B = dict()
    for r in U.groupby(B, group_keys=True).apply(lambda x: x).index:
        if r[0:-1] not in equiv_cls_B.keys():
            equiv_cls_B[r[0:-1]] = [r[-1]]
            continue
        equiv_cls_B[r[0:-1]].append(r[-1])
    return {k: set(equiv_cls_B[k]) for k in equiv_cls_B.keys()}

def under_appr(B, X) -> pd.DataFrame:
    under_appr_ind_set = set()
    X_ind_set = set(X.index)
    for cls_set in equiv_cls(B).values():
        if not cls_set.issubset(X_ind_set): continue
        under_appr_ind_set = under_appr_ind_set.union(cls_set)
    return X.loc[list(under_appr_ind_set)]

def pos(B) -> pd.DataFrame:
    _U_div_D = {tuple(d): U[(U[V]==d).all(axis=1)] for d in D}
    under_appr_dict = {k: under_appr(B, D_i) for k, D_i in _U_div_D.items()}
    return pd.concat(under_appr_dict.values())

def gamma(B): return len(pos(B)) / len(U)

In [13]:
from random import randint
B = []
for _ in range(7):
    B.append(C[randint(0, len(C)-1)])
B = list(set(B))
B

['Uniformity_of_cell_shape',
 'Marginal_adhesion',
 'Single_epithelial_cell_size',
 'Uniformity_of_cell_size',
 'Normal_nucleoli']

In [79]:
gamma(B)

0.9809663250366032